# Шапка

ML-HW06-v1 TEAM-11. translate-colab.
- Руденко Евгений Александрович
- Сажин Семён Михайлович
- Тихомиров Алексей Константинович

In [1]:
#!pip install -q --upgrade nltk gensim bokeh pandas

import nltk
nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\37727\AppData\Roaming\nltk_data...


[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\37727\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.manifold import TSNE


In [3]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from string import punctuation


## Запилим пословный машинный перевод!


In [4]:
# Команды Colab. На этой машине нет wget/unzip, а zip-архивы с Google Drive
# больше не отдаются, поэтому используются уже лежащие рядом файлы.
# !wget -O ukr_rus.train.txt -qq --no-check-certificate "https://drive.google.com/uc?export=download&id=1vAK0SWXUqei4zTimMvIhH3ufGPsbnC_O"
# !wget -O ukr_rus.test.txt -qq --no-check-certificate "https://drive.google.com/uc?export=download&id=1W9R2F8OeKHXruo2sicZ6FgBJUTJc8Us_"
# !wget -O fairy_tale.txt -qq --no-check-certificate "https://drive.google.com/uc?export=download&id=1sq8zSroFeg_afw-60OmY8RATdu_T1tej"
# !unzip cc.ru.300.vec.zip
# !unzip cc.uk.300.vec.zip

from pathlib import Path

needed = [
    "ukr_rus.train.txt",
    "ukr_rus.test.txt",
    "fairy_tale.txt",
    "cc.ru.300.vec",
    "cc.uk.300.vec",
]
missing = [name for name in needed if not Path(name).exists()]
if missing:
    raise FileNotFoundError("Рядом с ноутбуком нет файлов: " + ", ".join(missing))
print("данные на месте")


данные на месте


Напишем простенькую реализацию модели машинного перевода.

Идея основана на статье [Word Translation Without Parallel Data](https://arxiv.org/pdf/1710.04087.pdf). У авторов в репозитории еще много интересного: [https://github.com/facebookresearch/MUSE](https://github.com/facebookresearch/MUSE).

А мы будем переводить с украинского на русский.

![](https://raw.githubusercontent.com/yandexdataschool/nlp_course/master/resources/blue_cat_blue_whale.png)   
*синій кіт* vs. *синій кит*


In [5]:
from gensim.models import KeyedVectors

ru_emb = KeyedVectors.load_word2vec_format("cc.ru.300.vec")
uk_emb = KeyedVectors.load_word2vec_format("cc.uk.300.vec")


Посмотрим на пару серпень-август (являющихся переводом)


In [6]:
ru_emb.most_similar([ru_emb["август"]])


[('август', 1.0000001192092896),
 ('июль', 0.9383152723312378),
 ('сентябрь', 0.9240029454231262),
 ('июнь', 0.9222574830055237),
 ('октябрь', 0.9095539450645447),
 ('ноябрь', 0.8930036425590515),
 ('апрель', 0.8729087114334106),
 ('декабрь', 0.8652557730674744),
 ('март', 0.8545795679092407),
 ('февраль', 0.8401415944099426)]

In [7]:
uk_emb.most_similar([uk_emb["серпень"]])


[('серпень', 0.9999998807907104),
 ('липень', 0.9096441268920898),
 ('вересень', 0.9016969203948975),
 ('червень', 0.8992518782615662),
 ('жовтень', 0.8810408115386963),
 ('листопад', 0.8787633180618286),
 ('квітень', 0.8592804670333862),
 ('грудень', 0.8586863279342651),
 ('травень', 0.840811014175415),
 ('лютий', 0.8256431221961975)]

In [8]:
ru_emb.most_similar([uk_emb["серпень"]])


[('подход', 0.2230587750673294),
 ('подхода', 0.2142382562160492),
 ('Правовое', 0.20550373196601868),
 ('Профилактика', 0.20540718734264374),
 ('сторона', 0.20514370501041412),
 ('стиль', 0.2022884041070938),
 ('аспекте', 0.20203252136707306),
 ('системного', 0.20048706233501434),
 ('наиболее', 0.1991657316684723),
 ('Принцип', 0.1979515552520752)]

In [9]:
def load_word_pairs(filename):
    uk_ru_pairs = []
    uk_vectors = []
    ru_vectors = []
    with open(filename, "r", encoding='utf8') as inpf:
        for line in inpf:
            uk, ru = line.rstrip().split("\t")
            if uk not in uk_emb or ru not in ru_emb:
                continue
            uk_ru_pairs.append((uk, ru))
            uk_vectors.append(uk_emb[uk])
            ru_vectors.append(ru_emb[ru])
    return uk_ru_pairs, np.array(uk_vectors), np.array(ru_vectors)


uk_ru_train, X_train, Y_train = load_word_pairs("ukr_rus.train.txt")
uk_ru_test, X_test, Y_test = load_word_pairs("ukr_rus.test.txt")


### Учим маппинг из одного пространства эмбеддингов в другое

У нас есть пары слов, соответствующих друг другу, и их эмбеддинги. Найдем преобразование из одного пространства в другое, чтобы приблизить известные нам слова:

$$W^*= \arg\min_W ||WX - Y||_F, \text{где} ||*||_F - \text{норма Фробениуса}$$

Эта функция очень похожа на линейную регрессию (без биаса).

**Задание** Реализуйте её - воспользуйтесь `LinearRegression` из sklearn с `fit_intercept=False`:


In [10]:
from sklearn.linear_model import LinearRegression

mapping = LinearRegression(fit_intercept=False)
mapping.fit(X_train, Y_train)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",False
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float32](300, 300)","[[-0.08, 0.02, 0.03,..., 0.05, 0.1 , 0.04], [-0.05, 0.05, 0.04,..., 0.01, 0.04, 0.03], [ 0.14,-0.01, 0.04,...,-0.01, 0.05,-0.05], ..., [-0.01, 0.02,-0.05,...,-0.04, 0.06,-0.01], [ 0.06, 0.01, 0.03,..., 0.05, 0.06,-0.07], [-0.05,-0.03,-0.04,..., 0.03, 0.02,-0.04]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float,0
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,300
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(300)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float32](300,)","[17.34, 8.18, 7.57,..., 0.66, 0.63, 0.61]"


**Комментарий к заданию.** Линейная регрессия без свободного члена ищет матрицу, которая переводит украинские векторы в русские и минимизирует ошибку по всем координатам. В коде векторы лежат строками, поэтому обучается отображение $XW \approx Y$. Свободный член выключен: сдвиг всего облака не совмещает направления слов. «Серпень» после этого попадает к русским месяцам, но «август» ещё не первый сосед.


Проверим, куда перейдет `серпень`:


In [11]:
august = mapping.predict(uk_emb["серпень"].reshape(1, -1))
ru_emb.most_similar(august)


[('апрель', 0.8531431555747986),
 ('июнь', 0.8402522206306458),
 ('март', 0.838588297367096),
 ('сентябрь', 0.8331484198570251),
 ('февраль', 0.8311207294464111),
 ('октябрь', 0.8278018236160278),
 ('ноябрь', 0.8243727087974548),
 ('июль', 0.8229616284370422),
 ('август', 0.8112279176712036),
 ('январь', 0.8022984862327576)]

Должно получиться, что в топе содержатся разные месяцы, но август не первый.

Будем мерять percision top-k с k = 1, 5, 10.

**Задание** Реализуйте следующую функцию:


In [12]:
def precision(pairs, mapped_vectors, topn=1):
    """
    :args:
        pairs = list of right word pairs [(uk_word_0, ru_word_0), ...]
        mapped_vectors = list of embeddings after mapping from source embedding space to destination embedding space
        topn = the number of nearest neighbours in destination embedding space to choose from
    :returns:
        precision_val, float number, total number of words for those we can find right translation at top K.
    """
    assert len(pairs) == len(mapped_vectors)
    num_matches = 0
    for i, (_, ru) in enumerate(pairs):
        neighbours = ru_emb.most_similar([mapped_vectors[i]], topn=topn)
        if any(word == ru for word, _ in neighbours):
            num_matches += 1
    precision_val = num_matches / len(pairs)
    return precision_val


**Комментарий к заданию.** Precision@k — доля украинских слов, чей русский перевод попал в k ближайших соседей. В шаблоне сравнивалась длина с `ru_vectors`, но эта переменная есть только внутри `load_word_pairs`, поэтому проверка идёт по `mapped_vectors`. Без маппинга точность на тесте 0, на самих русских векторах она 1. У «серпень» «август» не входит в топ-5 и входит в топ-9. Линейный маппинг даёт precision@1 = 0.6565 и precision@5 = 0.8270.


In [13]:
assert precision([("серпень", "август")], august, topn=5) == 0.0
assert precision([("серпень", "август")], august, topn=9) == 1.0
assert precision([("серпень", "август")], august, topn=10) == 1.0


In [14]:
assert precision(uk_ru_test, X_test) == 0.0
assert precision(uk_ru_test, Y_test) == 1.0


In [15]:
precision_top1 = precision(uk_ru_test, mapping.predict(X_test), 1)
precision_top5 = precision(uk_ru_test, mapping.predict(X_test), 5)

assert precision_top1 >= 0.635
assert precision_top5 >= 0.813


### Улучшаем маппинг

Можно показать, что маппинг лучше строить ортогональным:
$$W^*= \arg\min_W ||WX - Y||_F \text{, где: } W^TW = I$$

Искать его можно через SVD:
$$X^TY=U\Sigma V^T\text{, singular value decompostion}$$

$$W^*=UV^T$$

**Задание** Реализуйте эту функцию.


In [16]:
def learn_transform(X_train, Y_train):
    """
    :returns: W* : float matrix[emb_dim x emb_dim] as defined in formulae above
    """
    u, _, vt = np.linalg.svd(np.matmul(X_train.T, Y_train), full_matrices=False)
    return np.matmul(u, vt)


**Комментарий к заданию.** Ортогональное преобразование — поворот без растяжения, поэтому соседства внутри языка сохраняются лучше, чем у обычной регрессии. Из SVD $X^T Y = U \Sigma V^T$ берётся $W = UV^T$. `np.linalg.svd` сразу возвращает $V^T$, в коде это `vt`. На тесте precision@1 = 0.6743 и precision@5 = 0.8321.


In [17]:
W = learn_transform(X_train, Y_train)


In [18]:
ru_emb.most_similar([np.matmul(uk_emb["серпень"], W)])


[('апрель', 0.8245131373405457),
 ('июнь', 0.8056630492210388),
 ('сентябрь', 0.8055761456489563),
 ('март', 0.8032935261726379),
 ('октябрь', 0.7987101674079895),
 ('июль', 0.7946797013282776),
 ('ноябрь', 0.793963611125946),
 ('август', 0.793819010257721),
 ('февраль', 0.7923861742019653),
 ('декабрь', 0.7715376019477844)]

In [19]:
assert precision(uk_ru_test, np.matmul(X_test, W)) >= 0.653
assert precision(uk_ru_test, np.matmul(X_test, W), 5) >= 0.824


### Пишем переводчик


Реализуем простой пословный переводчик - для каждого слова будем искать его ближайшего соседа в общем пространстве эмбеддингов. Если слова нет в эмбеддингах - просто копируем его.


In [20]:
with open("fairy_tale.txt", "r", encoding="utf-8") as inf:
    uk_sentences = [line.rstrip().lower() for line in inf]


In [21]:
def translate(sentence):
    """
    :args:
        sentence - sentence in Ukrainian (str)
    :returns:
        translation - sentence in Russian (str)

    * find ukrainian embedding for each word in sentence
    * transform ukrainian embedding vector
    * find nearest russian word and replace
    """
    translated = []
    for word in sentence.split(" "):
        if word not in uk_emb:
            translated.append(word)
            continue
        mapped = np.matmul(uk_emb[word], W)
        translated.append(ru_emb.most_similar([mapped], topn=1)[0][0])
    return " ".join(translated)


**Комментарий к заданию.** Фраза режется по пробелам. Если украинское слово есть в эмбеддингах, его вектор умножается на $W$ и заменяется ближайшим русским словом. Числа и знаки препинания копируются, поэтому `«.»` и `«1 , 3»` не меняются. `«кіт зловив мишу»` становится `«кот поймал мышку»`. На сказке это черновой подстрочник без грамматики.


In [22]:
assert translate(".") == "."
assert translate("1 , 3") == "1 , 3"
assert translate("кіт зловив мишу") == "кот поймал мышку"


In [23]:
for sentence in uk_sentences:
    print("src: {}\ndst: {}\n".format(sentence, translate(sentence)))


src: лисичка - сестричка і вовк - панібрат
dst: кошка – девочка и волк – панібрат

src: як була собі лисичка , да й пішла раз до однії баби добувать огню ; ввійшла у хату да й каже : " добрий день тобі , бабусю !
dst: как была себе кошка , че и пошла раз к обеих бабы пилить огня ; вошла во дом че и говорит : " хороший день тебе , маму !

src: дай мені огня " .
dst: дай мне огня " .

src: а баба тільки що вийняла із печі пирожок із маком , солодкий , да й положила , щоб він прохолов ; а лисичка се і підгледала , да тілько що баба нахилилась у піч , щоб достать огня , то лисичка зараз ухватила пирожок да і драла з хати , да , біжучи , весь мак із його виїла , а туда сміття наклала .
dst: а баба только что положила со печи пирог со мясом , сладкий , че и легла , чтобы он успокоиться ; а кошка ой и підгледала , че притом что баба упала во печь , чтобы фигура огня , то кошка сейчас ухватила пирог че и сунуть со изба , че , бежать , весь мак со его ворона , а туда мусор приняла .

src: прибі

src: тії хлопці так і зробили , а лисичка як зайшла за могилу , да зараз у ліс і повернула , щоб на дорозі не догнали ; прийшла у ліс да і зробила собі санки да й їде .
dst: бабы парни так и сделали , а кошка как зашла за могилу , че сейчас во лес и вернулась , чтобы по дороге не поймали ; пришла во лес че и сделала себе лыжи че и едет .

src: коли йде вовчик : " здорова була , лисичко - сестричко ! "
dst: когда идет заяц : " живая была , волк – сестра ! "

src: - " здоров , вовчику - братику ! "
dst: – " здоровье , волк – тебе ! "

src: - " де се ти узяла собі і бичка і санки ? "
dst: – " куда ой ты взяла себе и змея и лыжи ? "

src: - " е !
dst: – " ой !

src: зробила " .
dst: сделала " .

src: - " підвези ж і мене " .
dst: – " підвези же и меня " .

src: - " е , вовчику !
dst: – " ой , волк !

src: не можна " .
dst: не можно " .

src: - " мені хоть одну ніжку " .
dst: – " мне хоть одну ножка " .

src: - " одну можна " .
dst: – " одну можно " .

src: він і положив , да од'їхавши немн

src: се у мене був орішок , так я розкусив " .
dst: ой во меня был орішок , так мной схватил " .

src: да просить оп'ять , щоб і третю ногу положить ; лисичка і ту пустила , да тілько що оп'ять од'їхали , аж щось уже дужче тріснуло .
dst: че просит оп'ять , чтобы и третью ногу душу ; кошка и ту пустить , че притом что оп'ять од'їхали , до что-то уже сильнее стекло .

src: лисичка закричала : " ох , лишечко !
dst: кошка услышала : " ой , солнышко !

src: ти ж мені , братику , зовсім зламаєш санки " .
dst: ты же мне , тебе , совсем сломать лыжи " .

src: - " ні , лисичко , се я орішок розкусив " .
dst: – " ни , волк , ой мной орішок схватил " .

src: - " дай же і мені , бачиш який , що сам їж , а мені і не даєш " .
dst: – " дай то и мне , видишь который , что сам ел , а мне и не мне " .

src: - " нема уже більше , а я б дав " .
dst: – " нету уже больше , а мной бы дал " .

src: да і просить оп'ять , щоб пустила положить і послідню ногу .
dst: че и просит оп'ять , чтобы пустить душу и сес

src: да нарубай дерева , щоб нам оп'ять ізробить санки ; тільки рубавши кажи так : " рубайся ж , дерево , і криве і пряме " .
dst: че траву деревья , чтобы нам оп'ять ізробить лыжи ; только рубавши говори так : " рубайся же , дерево , и зеркало и прямое " .

src: він і пішов да й каже усе : " рубайся ж , дерево , усе пряме да пряме ! "
dst: он и пошел че и говорит всё : " рубайся же , дерево , всё прямое че прямое ! "

src: нарубавши і приносить ; лисичка увидала , що дерево не таке , як їй нужно , оп'ять розсердилась .
dst: нарубавши и приносит ; кошка увидала , что дерево не такое , как им надо , оп'ять подумала .

src: " ти , - говорить , - не казав , видно , так , як я тобі веліла ! "
dst: " ты , – говорит , – не говорил , видно , так , как мной тебе велел ! "

src: - " ні , я усе теє казав , що ти мені казала " .
dst: – " ни , мной всё Эх говорил , что ты мне говорила " .

src: - " да чомусь не таке рубалось ?
dst: – " че почему-то не такое рубалось ?

src: ну , сиди ж ти тут , а 

src: от вона встала , щоб поправить : може , що не так запряжено ; да , не хотячи , одоткнула солому , а оттуда так і сипнули горобці летіти .
dst: из она встала , чтобы решит : может , что не так лошади ; че , не дабы , одоткнула сено , а туда так и бросили звери лететь .

src: вона уже тоді побачила , що бичок неживий ; покинула його да й пішла .
dst: она уже тогда увидела , что заяц мертвец ; покинула его че и пошла .

src: легла на дорозі , аж дивиться - їде мужик з рибою ; вона і притворилась , що здохла .
dst: легла по дороге , до смотрит – едет мужик со рыбой ; она и притворилась , что свинья .

src: от мужик і говорить : " возьму я оцю лисицю , обдеру да хоть шапку собі зошью " .
dst: из мужик и говорит : " возьму мной мою собаку , обдеру че хоть шапка себе зошью " .

src: узяв да і положив ззаді у воза .
dst: взял че и положил сзади во коня .

src: вона замітила , що мужик не смотрить , стала ногами викидувать рибу з воза , а когда побачила , що навикидала уже багато , тоди по

src: - " як же ти ловила ? "
dst: – " как то ты поймал ? "

src: - " так , вовчику , уложила хвостик в ополонку , вожу тихенько да й кажу ; ловися , рибка , мала і велика !
dst: – " так , волк , уложила хвост во воду , беру тихо че и говорю ; ловися , рыба , имела и большая !

src: коли хочеш , то і ти піди , налови собі " .
dst: когда хочешь , то и ты иди , налови себе " .

src: він побіг да зробив так , як казала лисичка .
dst: он бросился че сделал так , как говорила кошка .

src: а лисичка стала за деревом да й дивиться ; коли у вовчика зовсім хвостик примерз , вона тоді побігла в село да й кричить : " ідіть , люди , вбивайте вовка ! "
dst: а кошка стала за деревом че и смотрит ; когда во медведя совсем хвост снег , она тогда пошла во село че и кричит : " идите , люди , убивать волк ! "

src: люди набігли з кольями да і убили його .
dst: люди бежали со кольями че и убили его .

src: 
dst: 



# Дополнительные материалы


## Почитать
### База:  
[On word embeddings - Part 1, Sebastian Ruder](http://ruder.io/word-embeddings-1/)  
[Deep Learning, NLP, and Representations, Christopher Olah](http://colah.github.io/posts/2014-07-NLP-RNNs-Representations/)  

### Как кластеризовать смыслы многозначных слов:  
[Making Sense of Word Embeddings (2016), Pelevina et al](http://anthology.aclweb.org/W16-1620)    

### Как оценивать эмбеддинги
[Evaluation methods for unsupervised word embeddings (2015), T. Schnabel](http://www.aclweb.org/anthology/D15-1036)  
[Intrinsic Evaluation of Word Vectors Fails to Predict Extrinsic Performance (2016), B. Chiu](https://www.aclweb.org/anthology/W/W16/W16-2501.pdf)  
[Problems With Evaluation of Word Embeddings Using Word Similarity Tasks (2016), M. Faruqui](https://arxiv.org/pdf/1605.02276.pdf)  
[Improving Reliability of Word Similarity Evaluation by Redesigning Annotation Task and Performance Measure (2016), Oded Avraham, Yoav Goldberg](https://arxiv.org/pdf/1611.03641.pdf)  
[Evaluating Word Embeddings Using a Representative Suite of Practical Tasks (2016), N. Nayak](https://cs.stanford.edu/~angeli/papers/2016-acl-veceval.pdf)  


## Посмотреть
[Word Vector Representations: word2vec, Lecture 2, cs224n](https://www.youtube.com/watch?v=ERibwqs9p38)
